In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- LOAD BALANCED DATA ---
# This must match the file name you just created
DATA_FILE = "../data/processed/AudioProcessed_3Class_Balanced.csv"

if not os.path.exists(DATA_FILE):
    print(f"❌ Error: File not found at {DATA_FILE}")
else:
    print(f"✅ Loading data from: {DATA_FILE}")
    df = pd.read_csv(DATA_FILE)

    # Check if data is actually balanced now
    print("\n📊 New Class Distribution (Should be ~1500 each):")
    print(df['label'].value_counts())

    # --- PREPARE DATA ---
    X = df.drop('label', axis=1)
    y = df['label']

✅ Loading data from: ../data/processed/AudioProcessed_3Class_Balanced.csv

📊 New Class Distribution (Should be ~1500 each):
label
2    1710
0    1146
1    1113
Name: count, dtype: int64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- SCALING ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- TRAIN ---
print("\n🚀 Training Gradient Boosting Model...")
gb_model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
gb_model.fit(X_train_scaled, y_train)

# --- EVALUATE ---
y_pred = gb_model.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
print(f"\n🏆 FINAL ACCURACY: {acc:.2%}")

print("\n--- Detailed Report ---")
print(classification_report(y_test, y_pred, target_names=['Hunger', 'Pain', 'Normal']))

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))


🚀 Training Gradient Boosting Model...

🏆 FINAL ACCURACY: 44.46%

--- Detailed Report ---
              precision    recall  f1-score   support

      Hunger       0.13      0.14      0.13       229
        Pain       0.35      0.34      0.34       223
      Normal       0.71      0.72      0.72       342

    accuracy                           0.44       794
   macro avg       0.40      0.40      0.40       794
weighted avg       0.44      0.44      0.44       794


--- Confusion Matrix ---
[[ 31 117  81]
 [129  75  19]
 [ 74  21 247]]


In [4]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- 1. LOAD DATA ---
# Use your balanced dataset
DATA_FILE = "../data/processed/AudioProcessed_3Class_Balanced.csv"

if not os.path.exists(DATA_FILE):
    print("❌ Error: Run data processing first!")
else:
    df = pd.read_csv(DATA_FILE)

    # --- 2. RELABEL FOR MODEL A (Pain vs. No Pain) ---
    # Current: 0=Hunger, 1=Pain, 2=Normal
    # Target:  1=Pain,   0=Others (Hunger + Normal)
    
    # Create a copy to avoid warnings
    df_model_a = df.copy()
    
    # If label is 1 (Pain), keep it 1. Everything else becomes 0.
    df_model_a['binary_label'] = df_model_a['label'].apply(lambda x: 1 if x == 1 else 0)

    print("📊 Model A Class Distribution:")
    print(df_model_a['binary_label'].value_counts()) 
    # You should see: 0 (~3000), 1 (~1500). This is okay.

    # --- 3. PREPARE ---
    X = df_model_a.drop(['label', 'binary_label'], axis=1) # Drop original label too
    y = df_model_a['binary_label']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # --- 4. SCALE ---
    scaler_a = StandardScaler()
    X_train_scaled = scaler_a.fit_transform(X_train)
    X_test_scaled = scaler_a.transform(X_test)

    # --- 5. TRAIN ---
    print("\n🚀 Training Model A (Pain vs No Pain)...")
    model_a = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
    model_a.fit(X_train_scaled, y_train)

    # --- 6. EVALUATE ---
    y_pred = model_a.predict(X_test_scaled)
    print(f"\n🏆 Model A Accuracy: {accuracy_score(y_test, y_pred):.2%}")
    print("\n--- Confusion Matrix (0=No Pain, 1=Pain) ---")
    print(confusion_matrix(y_test, y_pred))

    # --- 7. SAVE ---
    # Only save if accuracy is good (> 75%)
    SAVE_FOLDER = '../../../backEnd/mlModels/Cry/'
    os.makedirs(SAVE_FOLDER, exist_ok=True)
    
    joblib.dump(model_a, os.path.join(SAVE_FOLDER, 'model_a_pain.pkl'))
    joblib.dump(scaler_a, os.path.join(SAVE_FOLDER, 'scaler_a.pkl'))
    print("✅ Model A Saved!")

📊 Model A Class Distribution:
binary_label
0    2856
1    1113
Name: count, dtype: int64

🚀 Training Model A (Pain vs No Pain)...

🏆 Model A Accuracy: 63.85%

--- Confusion Matrix (0=No Pain, 1=Pain) ---
[[452 119]
 [168  55]]
✅ Model A Saved!


In [5]:
# --- 1. FILTER DATA FOR MODEL B ---
# We only want Hunger (0) and Normal (2)
df_model_b = df[df['label'] != 1].copy() # Drop Pain (1)

# Relabel: Hunger(0) -> 1, Normal(2) -> 0
# This makes "Hunger" the target we are looking for
df_model_b['binary_label'] = df_model_b['label'].apply(lambda x: 1 if x == 0 else 0)

print("📊 Model B Class Distribution:")
print(df_model_b['binary_label'].value_counts())

# --- 2. PREPARE ---
X = df_model_b.drop(['label', 'binary_label'], axis=1)
y = df_model_b['binary_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 3. SCALE ---
scaler_b = StandardScaler()
X_train_scaled = scaler_b.fit_transform(X_train)
X_test_scaled = scaler_b.transform(X_test)

# --- 4. TRAIN ---
print("\n🚀 Training Model B (Hunger vs Normal)...")
model_b = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
model_b.fit(X_train_scaled, y_train)

# --- 5. EVALUATE ---
y_pred = model_b.predict(X_test_scaled)
print(f"\n🏆 Model B Accuracy: {accuracy_score(y_test, y_pred):.2%}")

# --- 6. SAVE ---
joblib.dump(model_b, os.path.join(SAVE_FOLDER, 'model_b_hunger.pkl'))
joblib.dump(scaler_b, os.path.join(SAVE_FOLDER, 'scaler_b.pkl'))
print("✅ Model B Saved!")

📊 Model B Class Distribution:
binary_label
0    1710
1    1146
Name: count, dtype: int64

🚀 Training Model B (Hunger vs Normal)...

🏆 Model B Accuracy: 69.23%
✅ Model B Saved!
